Used the two cells below to download the information using yfinance 

In [ ]:
import yfinance as yf

# Define the ticker
ticker = 'NVDA'

# Download 1 year of daily data
nvidia_data = yf.download(ticker, period="2y", interval="1d", auto_adjust=False)

# Show the first few rows
print(nvidia_data.head())

[*********************100%***********************]  1 of 1 completed

Price       Adj Close      Close       High        Low       Open     Volume
Ticker           NVDA       NVDA       NVDA       NVDA       NVDA       NVDA
Date                                                                        
2023-06-26  40.607014  40.632000  42.764000  40.099998  42.460999  594322000
2023-06-27  41.850246  41.875999  41.939999  40.448002  40.799000  462175000
2023-06-28  41.091717  41.117001  41.845001  40.518002  40.660000  582639000
2023-06-29  40.796890  40.821999  41.599998  40.599998  41.557999  380514000
2023-06-30  42.275982  42.301998  42.549999  41.500999  41.680000  501148000


In [12]:
# Remove timezone from index
nvidia_data.index = nvidia_data.index.tz_localize(None)

nvidia_data.to_excel('nvidia_data.xlsx')

Some additional scripts for downloading all of the S&P500 data. Not really necessary for this project 

In [ ]:
import pandas as pd

# 1. Scrape the Wikipedia page for S&P 500 companies
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
table = pd.read_html(url, header=0)[0]

# 2. Extract the ticker symbols and convert any '.' to '-' for yfinance compatibility
sp500_tickers = table['Symbol'].str.replace('.', '-', regex=False).tolist()

# 3. (Optional) Print or inspect the first few tickers
print(sp500_tickers[:10])


['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A']


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.stats import kurtosis

# 2. Download data
data = yf.download(
    tickers=sp500_tickers,
    period="2y",
    interval="1d",
    auto_adjust=False,
    group_by='ticker',
    threads=True
)

# 3. Calculate log returns and kurtosis
kurtosis_dict = {}
for ticker in sp500_tickers:
    try:
        df = data[ticker]['Adj Close'].dropna()
        logret = np.log(df / df.shift(1)).dropna()
        # Pearson kurtosis
        kurt = kurtosis(logret, fisher=False)
        kurtosis_dict[ticker] = kurt
    except Exception:
        continue

# 4. Build DataFrame, sort, and show top 10
kurt_df = pd.DataFrame.from_dict(
    kurtosis_dict, orient='index', columns=['kurtosis']
).sort_values('kurtosis', ascending=False)

print("Top 10 stocks by kurtosis (most fat-tailed):")
print(kurt_df.head(10))

[*********************100%***********************]  503 of 503 completed


Top 10 stocks by kurtosis (most fat-tailed):
        kurtosis
GL    291.397213
EW    122.512141
WST   119.814523
PAYC  106.243450
DXCM  102.073015
JNPR   92.637268
DG     77.128515
ADM    75.985400
HII    70.892310
SRE    57.046805


In [4]:
print(data.head())

# Remove timezone from index
data.index = data.index.tz_localize(None)

data.to_excel('snp_data.xlsx')

Ticker           OTIS                                                       \
Price            Open       High        Low      Close  Adj Close   Volume   
Date                                                                         
2023-06-26  86.400002  87.383003  86.400002  87.120003  84.356239  1023200   
2023-06-27  87.500000  87.930000  86.949997  87.739998  84.956573  1052900   
2023-06-28  87.900002  87.900002  87.160004  87.510002  84.733864   962000   
2023-06-29  87.050003  88.639999  86.699997  88.610001  85.798965  1107000   
2023-06-30  89.339996  90.110001  89.010002  89.010002  86.186287  2776000   

Ticker           INTC                                   ...         VLO  \
Price            Open       High        Low      Close  ...         Low   
Date                                                    ...               
2023-06-26  33.189999  33.990002  33.099998  33.340000  ...  111.139999   
2023-06-27  33.220001  34.230000  33.009998  34.099998  ...  112.040001   


In [6]:
gl_data = data["GL"]

print(gl_data.to_string())

gl_data.to_excel('gl_data.xlsx')

Price             Open        High         Low       Close   Adj Close    Volume
Date                                                                            
2023-06-26  106.330002  107.260002  105.589996  107.019997  105.160912    487900
2023-06-27  107.349998  108.419998  107.199997  108.239998  106.359711    906100
2023-06-28  107.930000  108.110001  107.000000  107.449997  105.583427    583100
2023-06-29  107.769997  109.120003  107.769997  108.949997  107.057365    419900
2023-06-30  109.360001  110.089996  108.620003  109.620003  107.715729    577000
2023-07-03  108.849998  110.620003  108.559998  110.260002  108.567451    194900
2023-07-05  109.059998  109.860001  108.360001  109.639999  107.956970    503700
2023-07-06  109.260002  110.980003  108.669998  110.949997  109.246872    662100
2023-07-07  111.160004  112.750000  111.160004  111.660004  109.945969    553900
2023-07-10  111.519997  112.370003  110.930000  111.760002  110.044449    424100
2023-07-11  112.019997  112.